# Notebook 1/3 — Setup & Ingestion (demo data)

Goal: **generate a synthetic dataset** matching the client model (`Dossier` + identifiers) and **ingest** it into Neo4j.

> This notebook is only useful for the demo. In production, the `Dossier`/identifiers graph is already populated by the existing pipelines — the client starts directly from Notebooks 2 and 3.

## Data model

```
(:Dossier {NODOS, DATE_COMMANDE, TOP_FRAUDE, ...})
   -[:EMAIL_ROOT]->(:EmailRoot)
   -[:TELEPHONE]->(:Telephone)
   -[:ADRESSE_IP]->(:AdresseIP)
   -[:DEVICE]->(:DeviceID)
   -[:CARTE_BANCAIRE]->(:CarteBancaire)
   -[:COMPTE]->(:Compte)
   -[:ETAT_CIVIL]->(:EtatCivil)
   -[:ADRESSE_EMPRUNTEUR]->(:AdressePostale)
   -[:FOYER_SICLID]->(:FoyerSiclid)
   -[:PANIER]->(:Panier {MONTANT...})
```

Two `Dossier` nodes that **share an identifier** are implicitly linked → they form a **community** (weakly connected component).

## Flow of the 3 notebooks

1. **`01_setup_ingestion.ipynb`** (this notebook) — generates and ingests the demo data.
2. **`02_build_relations.ipynb`** — builds `SIMILARITE` then the temporal structure (prerequisite for the logic).
3. **`03_training_walkforward.ipynb`** — *walk-forward* training day by day, without leaking the future.


In [6]:
# Cell 1: Imports & configuration

!pip install neo4j pandas numpy -q

import numpy as np
import pandas as pd
import random
import time
from datetime import datetime, timedelta
from collections import defaultdict
from neo4j import GraphDatabase

# --- Neo4j connection ---
NEO4J_URI = "bolt://localhost:7687"
NEO4J_USER = "neo4j"
NEO4J_PASSWORD = "password"
NEO4J_DATABASE = "fraudwcctemporal"

# --- Generation parameters ---
N_DOSSIERS = 1_000_000   # volume (drop to 200_000 for a quick test)
DAYS = 730             # spread DATE_COMMANDE over ~2 years (required for the 7D/7M/1Y windows)
SEED = 42

print("✅ Imports OK")


✅ Imports OK


In [7]:
# Cell 2: Generate synthetic data matching the client model
#
# Goal: reproduce a REALISTIC community-size distribution (cf. prod stats):
#   - median size 1 (vast majority of "solo" dossiers)
#   - a few VERY large communities (up to ~680 dossiers)
# SHARED identifiers create the communities (two dossiers sharing a strong
# identifier are linked):
#   - solo          : all identifiers unique -> community of size 1
#   - famille       : small group sharing Foyer + EtatCivil + Adresse (legitimate)
#   - fraude_anneau : group sharing a STRONG identifier (email / device / CB / IBAN)
#   - fraude_grande : very large ring sharing a strong identifier -> large community

# Types d'identifiants : relation -> (label, propriété-clé)
IDENTIFIERS = {
    "EMAIL_ROOT":         ("EmailRoot",      "ROOT_EMAIL"),
    "TELEPHONE":          ("Telephone",      "TITTELEPHONEPORTABLE"),
    "ADRESSE_IP":         ("AdresseIP",      "IP"),
    "DEVICE":             ("DeviceID",       "DEVICE_ID"),
    "CARTE_BANCAIRE":     ("CarteBancaire",  "DISTRIBCLEHASHCB"),
    "COMPTE":             ("Compte",         "ACCOUNTNUMBERIBAN"),
    "ETAT_CIVIL":         ("EtatCivil",      "NOM_PRENOM_DDN"),
    "ADRESSE_EMPRUNTEUR": ("AdressePostale", "ADRESSE_CP_VILLE_CLEAN_PAYS"),
    "FOYER_SICLID":       ("FoyerSiclid",    "NUM_FOYER"),
}

STRONG = ["EMAIL_ROOT", "DEVICE", "CARTE_BANCAIRE", "COMPTE"]  # strong identifiers (fraud rings)
VILLES = ["Paris", "Lyon", "Marseille", "Lille", "Bordeaux", "Toulouse", "Nantes", "Nice"]
FEUX = ["VERT", "ORANGE", "ROUGE"]

# --- Tuning the community-size distribution ---
N_GRANDES = 25                    # number of very large communities (like prod: ~678, 416, ...)
GRANDE_MIN, GRANDE_MAX = 150, 680  # size bounds for the very large communities
PART_ANNEAUX = 0.13               # fraction of dossiers in medium rings (6-40)
PART_FAMILLES = 0.15              # fraction of dossiers in families (2-5)
# the rest -> solos (size 1), which keeps a median of 1 as in prod

# --- Temporal burst (to create a real "acceleration" signal) ---
# Only SOME fraud rings are bursty: their dossiers are concentrated in a short
# window instead of being spread uniformly over the 2 years. This is what makes
# community-acceleration features (density / accel_7_30) actually discriminative,
# while non-bursty rings stay indistinguishable in time (realistic mix).
PART_BURST_RINGS = 0.5                    # fraction of fraud rings that are bursty
BURST_MIN_DAYS, BURST_MAX_DAYS = 2, 21    # window over which a bursty ring is concentrated


def _build_community_plan(n_dossiers, rng):
    """List of communities: (size, link type, fraudulent?, tag, bursty?)."""
    plan, covered = [], 0
    for _ in range(N_GRANDES):                       # 1) very large communities
        s = int(rng.integers(GRANDE_MIN, GRANDE_MAX + 1))
        bursty = bool(rng.random() < PART_BURST_RINGS)
        plan.append((s, str(rng.choice(STRONG)), True, "fraude_grande", bursty))
        covered += s
    while covered < PART_ANNEAUX * n_dossiers:       # 2) medium fraud rings
        s = int(rng.integers(6, 41))
        bursty = bool(rng.random() < PART_BURST_RINGS)
        plan.append((s, str(rng.choice(STRONG)), True, "fraude_anneau", bursty))
        covered += s
    while covered < (PART_ANNEAUX + PART_FAMILLES) * n_dossiers:  # 3) legitimate families
        s = int(rng.integers(2, 6))
        plan.append((s, "FOYER_SICLID", False, "famille", False))
        covered += s
    for _ in range(max(0, n_dossiers - covered)):    # 4) solos
        plan.append((1, None, False, "solo", False))
    rng.shuffle(plan)
    return plan


def generate_credit_dataset(n_dossiers, days, seed):
    rng = np.random.default_rng(seed)
    print(f"🚀 Generating {n_dossiers:,} dossiers over {days} days")
    start = time.time()

    end_time = datetime.now() - timedelta(hours=1)
    start_time = end_time - timedelta(days=days)
    range_sec = int((end_time - start_time).total_seconds())

    plan = _build_community_plan(n_dossiers, rng)

    rows, i = [], 0
    comm_meta = {}  # cid -> {link_type, shared, size, max_ts}: strong fraud rings (for MERGE cases)
    for cid, (size, link_type, is_fraud_comm, tag, bursty) in enumerate(plan):
        if i >= n_dossiers:
            break
        shared = f"{tag}_{cid}"  # SHARED identifier value within the community
        if is_fraud_comm and link_type in STRONG:
            comm_meta[cid] = {"link_type": link_type, "shared": shared, "size": size, "max_ts": start_time}
        # Bursty ring -> all its dossiers fall inside a short [anchor, anchor+burst] window
        if bursty:
            burst_sec = int(rng.integers(BURST_MIN_DAYS, BURST_MAX_DAYS + 1)) * 86_400
            anchor_sec = int(rng.integers(0, max(1, range_sec - burst_sec)))
        for _ in range(size):
            if i >= n_dossiers:
                break
            # unique identifiers by default
            ident = {rel: f"{rel.lower()}_{i}" for rel in IDENTIFIERS}
            if link_type is not None:
                ident[link_type] = f"{link_type.lower()}_{shared}"
                if tag == "famille":  # a household shares several axes
                    ident["ETAT_CIVIL"] = f"etat_civil_{shared}"
                    ident["ADRESSE_EMPRUNTEUR"] = f"adresse_emprunteur_{shared}"

            fraude = bool(rng.random() < (0.85 if is_fraud_comm else 0.004))
            if bursty:  # concentrated in time -> creates the acceleration signal
                ts = start_time + timedelta(seconds=anchor_sec + int(rng.integers(0, burst_sec)))
            else:       # spread uniformly over the whole period (no burst)
                ts = start_time + timedelta(seconds=int(rng.integers(0, range_sec)))
            if cid in comm_meta and ts > comm_meta[cid]["max_ts"]:
                comm_meta[cid]["max_ts"] = ts
            rows.append({
                "nodos": f"DOS_{i:08d}",
                "date_commande": ts.isoformat(),
                "pattern": tag,
                "top_fraude": fraude,
                "fpd": bool(fraude and rng.random() < 0.6),
                "spd": bool(fraude and rng.random() < 0.4),
                "tpd": bool(fraude and rng.random() < 0.3),
                "feu_initial": str(rng.choice(FEUX)),
                "feu_final": "ROUGE" if (fraude and rng.random() < 0.5) else str(rng.choice(FEUX)),
                "flag_refuse": bool(fraude and rng.random() < 0.5),
                "ville": str(rng.choice(VILLES)),
                "montant": float(round(rng.gamma(3.0, 1500.0) + 300, 2)),
                "email": ident["EMAIL_ROOT"],
                "tel": ident["TELEPHONE"],
                "ip": ident["ADRESSE_IP"],
                "device": ident["DEVICE"],
                "cb": ident["CARTE_BANCAIRE"],
                "compte": ident["COMPTE"],
                "etatcivil": ident["ETAT_CIVIL"],
                "adresse": ident["ADRESSE_EMPRUNTEUR"],
                "foyer": ident["FOYER_SICLID"],
            })
            i += 1
            if i % 200_000 == 0:
                print(f"   {i:,} dossiers ({time.time()-start:.1f}s)")

    # --- MERGE cases: bridge dossiers connecting TWO fraud rings -----------------
    # Each "bridge" shares a STRONG identifier with TWO different rings at once, so
    # the two communities merge into one. This is the ONLY topology that makes the
    # COMPONENT_PARENT forest BRANCH (a merge node, in-degree 2) -- it never happens
    # with the single-shared-id rings above. Used to test the point-in-time /
    # windowed-community logic on non-linear (branching) communities.
    N_FUSIONS = 30
    eligible = [m for m in comm_meta.values() if m["size"] >= 3]
    rng.shuffle(eligible)
    used = [False] * len(eligible)
    n_fusions = 0
    for x in range(len(eligible)):  # bridges are ADDED on top of n_dossiers (a handful)
        if n_fusions >= N_FUSIONS:
            break
        if used[x]:
            continue
        a = eligible[x]
        # partner ring with a DIFFERENT strong id (else both shared values would
        # collide on the same identifier field and no cross-lineage bridge is made)
        y = next((j for j in range(len(eligible))
                  if not used[j] and j != x and eligible[j]["link_type"] != a["link_type"]), None)
        if y is None:
            continue
        b = eligible[y]
        used[x] = used[y] = True

        ident = {rel: f"{rel.lower()}_{i}" for rel in IDENTIFIERS}
        ident[a["link_type"]] = f'{a["link_type"].lower()}_{a["shared"]}'
        ident[b["link_type"]] = f'{b["link_type"].lower()}_{b["shared"]}'
        # dated AFTER both rings -> the bridge is the newest of the merged component,
        # so it gets a SIMILARITE predecessor in EACH ring -> in-degree 2 (a merge node)
        base = max(a["max_ts"], b["max_ts"])
        ts = min(base + timedelta(seconds=int(rng.integers(3_600, 7 * 86_400))), end_time)
        fraude = bool(rng.random() < 0.85)
        rows.append({
            "nodos": f"DOS_{i:08d}",
            "date_commande": ts.isoformat(),
            "pattern": "fraude_fusion",
            "top_fraude": fraude,
            "fpd": bool(fraude and rng.random() < 0.6),
            "spd": bool(fraude and rng.random() < 0.4),
            "tpd": bool(fraude and rng.random() < 0.3),
            "feu_initial": str(rng.choice(FEUX)),
            "feu_final": "ROUGE" if (fraude and rng.random() < 0.5) else str(rng.choice(FEUX)),
            "flag_refuse": bool(fraude and rng.random() < 0.5),
            "ville": str(rng.choice(VILLES)),
            "montant": float(round(rng.gamma(3.0, 1500.0) + 300, 2)),
            "email": ident["EMAIL_ROOT"],
            "tel": ident["TELEPHONE"],
            "ip": ident["ADRESSE_IP"],
            "device": ident["DEVICE"],
            "cb": ident["CARTE_BANCAIRE"],
            "compte": ident["COMPTE"],
            "etatcivil": ident["ETAT_CIVIL"],
            "adresse": ident["ADRESSE_EMPRUNTEUR"],
            "foyer": ident["FOYER_SICLID"],
        })
        i += 1
        n_fusions += 1

    dossiers_df = pd.DataFrame(rows)
    tailles = [s for s, _, _, _, _ in plan if s > 1]
    n_burst = sum(1 for _, _, _, _, b in plan if b)
    print(f"✅ {len(dossiers_df):,} dossiers generated in {time.time()-start:.1f}s")
    print(dossiers_df["pattern"].value_counts())
    print(f"   Communities > 1 : {len(tailles):,} | target max size : {max(tailles) if tailles else 1}")
    print(f"   Bursty fraud rings : {n_burst:,} (concentrated in time -> acceleration signal)")
    print(f"   Merge (bridge) dossiers : {n_fusions:,} (each merges 2 rings -> branching component)")
    print(f"   Labeled fraud rate : {dossiers_df['top_fraude'].mean():.2%}")
    return dossiers_df


dossiers_df = generate_credit_dataset(N_DOSSIERS, DAYS, SEED)
dossiers_df.head()


🚀 Generating 1,000,000 dossiers over 730 days
   200,000 dossiers (3.3s)
   400,000 dossiers (6.3s)
   600,000 dossiers (9.1s)
   800,000 dossiers (12.1s)
   1,000,000 dossiers (14.9s)
✅ 1,000,030 dossiers generated in 16.5s
pattern
solo             719999
famille          149998
fraude_anneau    119792
fraude_grande     10211
fraude_fusion        30
Name: count, dtype: int64
   Communities > 1 : 48,042 | target max size : 649
   Bursty fraud rings : 2,631 (concentrated in time -> acceleration signal)
   Merge (bridge) dossiers : 30 (each merges 2 rings -> branching component)
   Labeled fraud rate : 11.42%


,nodos,date_commande,pattern,top_fraude,fpd,spd,tpd,feu_initial,feu_final,flag_refuse,...,montant,email,tel,ip,device,cb,compte,etatcivil,adresse,foyer
0,DOS_00000000,2024-09-24T21:52:30.185081,solo,False,False,False,False,ROUGE,VERT,False,...,2121.46,email_root_0,telephone_0,adresse_ip_0,device_0,carte_bancaire_0,compte_0,etat_civil_0,adresse_emprunteur_0,foyer_siclid_0
1,DOS_00000001,2024-11-24T23:08:43.185081,solo,False,False,False,False,ROUGE,VERT,False,...,6975.27,email_root_1,telephone_1,adresse_ip_1,device_1,carte_bancaire_1,compte_1,etat_civil_1,adresse_emprunteur_1,foyer_siclid_1
2,DOS_00000002,2025-03-27T15:55:16.185081,solo,False,False,False,False,VERT,ROUGE,False,...,8110.56,email_root_2,telephone_2,adresse_ip_2,device_2,carte_bancaire_2,compte_2,etat_civil_2,adresse_emprunteur_2,foyer_siclid_2
3,DOS_00000003,2025-06-20T03:34:24.185081,solo,False,False,False,False,VERT,VERT,False,...,8484.77,email_root_3,telephone_3,adresse_ip_3,device_3,carte_bancaire_3,compte_3,etat_civil_3,adresse_emprunteur_3,foyer_siclid_3
4,DOS_00000004,2025-04-24T05:54:44.185081,solo,False,False,False,False,ORANGE,ORANGE,False,...,9433.06,email_root_4,telephone_4,adresse_ip_4,device_4,carte_bancaire_4,compte_4,etat_civil_4,adresse_emprunteur_4,foyer_siclid_4


In [8]:
# Cell 3: Neo4j connection + schema (uniqueness constraints = indexes to speed up MERGE)

driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USER, NEO4J_PASSWORD))
driver.verify_connectivity()
print("✅ Connected to Neo4j")

CONSTRAINTS = [
    "CREATE CONSTRAINT dossier_nodos IF NOT EXISTS FOR (d:Dossier) REQUIRE d.NODOS IS UNIQUE",
    "CREATE CONSTRAINT email_root IF NOT EXISTS FOR (n:EmailRoot) REQUIRE n.ROOT_EMAIL IS UNIQUE",
    "CREATE CONSTRAINT telephone IF NOT EXISTS FOR (n:Telephone) REQUIRE n.TITTELEPHONEPORTABLE IS UNIQUE",
    "CREATE CONSTRAINT adresse_ip IF NOT EXISTS FOR (n:AdresseIP) REQUIRE n.IP IS UNIQUE",
    "CREATE CONSTRAINT device_id IF NOT EXISTS FOR (n:DeviceID) REQUIRE n.DEVICE_ID IS UNIQUE",
    "CREATE CONSTRAINT carte_bancaire IF NOT EXISTS FOR (n:CarteBancaire) REQUIRE n.DISTRIBCLEHASHCB IS UNIQUE",
    "CREATE CONSTRAINT compte IF NOT EXISTS FOR (n:Compte) REQUIRE n.ACCOUNTNUMBERIBAN IS UNIQUE",
    "CREATE CONSTRAINT etat_civil IF NOT EXISTS FOR (n:EtatCivil) REQUIRE n.NOM_PRENOM_DDN IS UNIQUE",
    "CREATE CONSTRAINT adresse_postale IF NOT EXISTS FOR (n:AdressePostale) REQUIRE n.ADRESSE_CP_VILLE_CLEAN_PAYS IS UNIQUE",
    "CREATE CONSTRAINT foyer_siclid IF NOT EXISTS FOR (n:FoyerSiclid) REQUIRE n.NUM_FOYER IS UNIQUE",
    "CREATE CONSTRAINT panier IF NOT EXISTS FOR (n:Panier) REQUIRE n.CODDOSSIERSOUSCRIPTION IS UNIQUE",
    "CREATE INDEX dossier_date IF NOT EXISTS FOR (d:Dossier) ON (d.DATE_COMMANDE)",
]

with driver.session(database=NEO4J_DATABASE) as session:
    for c in CONSTRAINTS:
        session.run(c)
print("✅ Constraints & indexes in place")


✅ Connected to Neo4j
✅ Constraints & indexes in place


In [9]:
# Cell 4: Batch ingestion (Dossier + identifiers + bipartite relationships). NO SAME_CC_AS.

INGEST_QUERY = """
UNWIND $rows AS row
CREATE (d:Dossier {NODOS: row.nodos})
SET d.DATE_COMMANDE = datetime(row.date_commande),
    d.pattern       = row.pattern,
    d.TOP_FRAUDE    = row.top_fraude,
    d.FPD           = row.fpd,
    d.SPD           = row.spd,
    d.TPD           = row.tpd,
    d.FEU_INITIAL   = row.feu_initial,
    d.FEU_FINAL     = row.feu_final,
    d.Flag_refuse   = row.flag_refuse,
    d.VILLE_EMPRUNTEUR = row.ville
MERGE (em:EmailRoot {ROOT_EMAIL: row.email})              MERGE (d)-[:EMAIL_ROOT]->(em)
MERGE (te:Telephone {TITTELEPHONEPORTABLE: row.tel})       MERGE (d)-[:TELEPHONE]->(te)
MERGE (ip:AdresseIP {IP: row.ip})                          MERGE (d)-[:ADRESSE_IP]->(ip)
MERGE (dv:DeviceID {DEVICE_ID: row.device})                MERGE (d)-[:DEVICE]->(dv)
MERGE (cb:CarteBancaire {DISTRIBCLEHASHCB: row.cb})        MERGE (d)-[:CARTE_BANCAIRE]->(cb)
MERGE (co:Compte {ACCOUNTNUMBERIBAN: row.compte})          MERGE (d)-[:COMPTE]->(co)
MERGE (ec:EtatCivil {NOM_PRENOM_DDN: row.etatcivil})       MERGE (d)-[:ETAT_CIVIL]->(ec)
MERGE (ad:AdressePostale {ADRESSE_CP_VILLE_CLEAN_PAYS: row.adresse}) MERGE (d)-[:ADRESSE_EMPRUNTEUR]->(ad)
MERGE (fo:FoyerSiclid {NUM_FOYER: row.foyer})              MERGE (d)-[:FOYER_SICLID]->(fo)
MERGE (pa:Panier {CODDOSSIERSOUSCRIPTION: row.nodos})
SET pa.MONTANTCREDITCLASSIQUEAPRESSE = row.montant
MERGE (d)-[:PANIER]->(pa)
"""

BATCH_SIZE = 2_000
records = dossiers_df.to_dict("records")

print(f"🚀 Ingesting {len(records):,} dossiers (batch={BATCH_SIZE})")
start = time.time()
with driver.session(database=NEO4J_DATABASE) as session:
    for i in range(0, len(records), BATCH_SIZE):
        batch = records[i:i + BATCH_SIZE]
        session.run(INGEST_QUERY, rows=batch)
        if (i + BATCH_SIZE) % 40_000 == 0:
            done = min(i + BATCH_SIZE, len(records))
            print(f"   {done:,}/{len(records):,} ({time.time()-start:.1f}s)")

print(f"✅ Ingestion finished in {(time.time()-start)/60:.1f} min")


🚀 Ingesting 1,000,030 dossiers (batch=2000)
   40,000/1,000,030 (7.2s)
   80,000/1,000,030 (14.5s)
   120,000/1,000,030 (20.9s)
   160,000/1,000,030 (26.5s)
   200,000/1,000,030 (31.6s)
   240,000/1,000,030 (36.9s)
   280,000/1,000,030 (43.1s)
   320,000/1,000,030 (50.4s)
   360,000/1,000,030 (56.8s)
   400,000/1,000,030 (64.3s)
   440,000/1,000,030 (69.8s)
   480,000/1,000,030 (75.6s)
   520,000/1,000,030 (80.8s)
   560,000/1,000,030 (86.9s)
   600,000/1,000,030 (94.1s)
   640,000/1,000,030 (100.9s)
   680,000/1,000,030 (107.2s)
   720,000/1,000,030 (112.5s)
   760,000/1,000,030 (118.2s)
   800,000/1,000,030 (124.1s)
   840,000/1,000,030 (131.0s)
   880,000/1,000,030 (138.0s)
   920,000/1,000,030 (144.2s)
   960,000/1,000,030 (150.2s)
   1,000,000/1,000,030 (155.4s)
✅ Ingestion finished in 2.6 min


In [10]:
# Cell 5: Verification

with driver.session(database=NEO4J_DATABASE) as session:
    counts = session.run("""
        MATCH (d:Dossier)
        RETURN count(d) AS dossiers,
               sum(CASE WHEN d.TOP_FRAUDE THEN 1 ELSE 0 END) AS fraudes
    """).single()
    rels = session.run("""
        MATCH ()-[r]->()
        RETURN type(r) AS type, count(r) AS n ORDER BY n DESC
    """).data()

print(f"Dossiers : {counts['dossiers']:,}  |  Labeled fraud : {counts['fraudes']:,}")
print("Relationships :")
for r in rels:
    print(f"   {r['type']:22s} {r['n']:>12,}")

print("\n👉 Go to Notebook 2 to build SIMILARITE + the point-in-time structure.")


Dossiers : 1,000,030  |  Labeled fraud : 114,164
Relationships :
   EMAIL_ROOT                1,000,030
   TELEPHONE                 1,000,030
   ADRESSE_IP                1,000,030
   DEVICE                    1,000,030
   CARTE_BANCAIRE            1,000,030
   COMPTE                    1,000,030
   ETAT_CIVIL                1,000,030
   ADRESSE_EMPRUNTEUR        1,000,030
   FOYER_SICLID              1,000,030
   PANIER                    1,000,030

👉 Go to Notebook 2 to build SIMILARITE + the point-in-time structure.
